In [21]:
import pandas as pd
import math
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
from typing import List
from enum import Enum
from dataclasses import dataclass
from typing import List

### Data Loading

Load the market data CSV file into a pandas DataFrame.

**Requirements:**
- Data must be at **minute-level granularity** (1-minute OHLCV bars).
- Only a **single ticker** is supported at this time.
- The CSV must contain the following columns (in any order): `Datetime`, `Close`, `High`, `Low`, `Open`, `Volume`

> These column names are required exactly as shown because the `FeatureEngineering` class references them by name when computing technical indicators.


In [22]:
filename = 'market_data.csv'
data = pd.read_csv(filename)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7020 entries, 0 to 7019
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  7020 non-null   int64  
 1   Datetime    7020 non-null   object 
 2   Close       7020 non-null   float64
 3   High        7020 non-null   float64
 4   Low         7020 non-null   float64
 5   Open        7020 non-null   float64
 6   Volume      7020 non-null   int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 384.0+ KB


### Feature Engineering

The `FeatureEngineering` class enriches the raw OHLCV data with technical indicators, volume metrics, return-based features, time features, and a binary classification target. All features are added **in-place** to the internal DataFrame.

Calling `add_features()` runs the full pipeline in this order:

1. **`add_indicators()`** — Price-based momentum and trend indicators (SMA, EMA, MACD, RSI)
2. **`add_volume_indicators()`** — Volatility and volume flow indicators (ATR, Volume SMA, OBV)
3. **`add_time_features()`** — Log returns, lagged returns, and calendar/time-of-day features
4. **`add_target()`** — Binary label: `1` if Close > previous Close (up), else `0` (down)

After running `add_features()`, the dataset will have **35 columns** total (7 original + 28 engineered).

> **Note:** Some indicators require a warm-up period (e.g., SMA-50 needs 50 bars), so the first rows will contain `NaN` values. These are dropped during preprocessing.


In [23]:
class FeatureEngineering():
    def __init__(self, data):
        self.data = data

    def get_data(self):
        return self.data

    def add_features(self):
        self.add_indicators()
        self.add_volume_indicators()
        self.add_time_features()
        self.add_target()

    def add_indicators(self):
        close = self.data['Close']

        # SMA (5, 20, 50)
        for period in [5, 20, 50]:
            self.data[f"sma_{period}"] = close.rolling(window=period).mean()

        # EMA (5, 10, 20, 50)
        for period in [5, 10, 20, 50]:
            self.data[f"ema_{period}"] = close.ewm(span=period, adjust=False).mean()

        # MACD(12, 26, 9)
        ema_12 = close.ewm(span=12, adjust=False).mean()
        ema_26 = close.ewm(span=26, adjust=False).mean()
        self.data["macd"] = ema_12 - ema_26
        self.data["macd_signal"] = self.data["macd"].ewm(span=9, adjust=False).mean()
        self.data["macd_hist"] = self.data["macd"] - self.data["macd_signal"]

        # RSI
        self.data["rsi_14"] = self.compute_rsi(close, 14)
        self.data["rsi_7"] = self.compute_rsi(close, 7)

    def compute_rsi(self, series: pd.Series, period: int) -> pd.Series:
        delta = series.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        avg_gain = gain.ewm(com=period - 1, adjust=False).mean() # Wilder smoothing
        avg_loss = loss.ewm(com=period - 1, adjust=False).mean()
        rs = avg_gain / avg_loss
        return 100 - (100 / (1 + rs))

    def add_volume_indicators(self):
        high, low, close, volume = self.data["High"], self.data["Low"], self.data["Close"], self.data['Volume']
        prev_close = close.shift(1)

        # ATR (14)
        true_range = pd.concat([
            high - low,                  # Current high - low
            (high - prev_close).abs(),   # |High - Previous close|
            (low - prev_close).abs(),    # |Low - Previous close|
        ], axis=1).max(axis=1)

        self.data["atr_14"] = true_range.ewm(com=13, adjust=False).mean()

        # Volume SMA (5, 20)
        for period in [5, 20]:
            self.data[f"volume_sma_{period}"] = volume.rolling(window=period).mean()

        # OBV
        direction = pd.Series(0, index=self.data.index)
        direction[close > prev_close] = 1
        direction[close < prev_close] = -1

        self.data["obv"] = (direction * volume).cumsum()

    def add_time_features(self):
        close = self.data['Close']

        # Log Returns (1, 3, 5, 20)
        for period in [1, 3, 5, 20]:
            self.data[f"return_{period}"] = (close / close.shift(period)).apply(lambda x: math.log(x))

        # Lagged Returns
        ret_1 = self.data["return_1"]
        for lag in [1, 3, 5, 20]:
            self.data[f"lagged_return_{lag}"] = ret_1.shift(lag)

        # Time Features
        t = pd.to_datetime(self.data["Datetime"])
        self.data["hour_of_day"] = t.dt.hour
        self.data["minute_of_hour"] = t.dt.minute
        self.data["day_of_week"] = t.dt.dayofweek + 1

    def add_target(self):
        self.data['trend'] = (self.data["Close"] > self.data["Close"].shift(1)).astype(int)


In [24]:
feature_engineering = FeatureEngineering(data)
feature_engineering.add_features()

fe_data = feature_engineering.get_data()
fe_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7020 entries, 0 to 7019
Data columns (total 35 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        7020 non-null   int64  
 1   Datetime          7020 non-null   object 
 2   Close             7020 non-null   float64
 3   High              7020 non-null   float64
 4   Low               7020 non-null   float64
 5   Open              7020 non-null   float64
 6   Volume            7020 non-null   int64  
 7   sma_5             7016 non-null   float64
 8   sma_20            7001 non-null   float64
 9   sma_50            6971 non-null   float64
 10  ema_5             7020 non-null   float64
 11  ema_10            7020 non-null   float64
 12  ema_20            7020 non-null   float64
 13  ema_50            7020 non-null   float64
 14  macd              7020 non-null   float64
 15  macd_signal       7020 non-null   float64
 16  macd_hist         7020 non-null   float64


### Preprocessing

The `Preprocess` class cleans the dataset and produces chronological train/test splits for both model types.

**Steps performed on initialization:**
- Drops all rows containing `NaN` values (produced by rolling/lagged indicators during the warm-up period).
- Resets the DataFrame index.

**Train/test split strategy:**
- Uses an **80/20 chronological split** — the first 80% of rows become the training set and the remaining 20% become the test set.
- A random shuffle is intentionally avoided to prevent data leakage from future bars into the training set.

**Two split methods are available:**

| Method | Target (`y`) | Use case |
|---|---|---|
| `get_train_test_clf()` | `trend` (0/1) | Random Forest Classifier — predicts up/down direction |
| `get_train_test_reg()` | `Close` (float) | Random Forest Regressor — predicts the next close price |

In both cases, `Datetime`, `Close`, and `trend` are excluded from the feature matrix `X` to avoid target leakage.


In [25]:
class Preprocess():
    def __init__(self, data):
        # drop NaN rows
        data = data.dropna().reset_index(drop=True)
        self.data = data

    def get_data(self):
        return self.data

    def get_train_test_clf(self):
        drop_cols = ['trend', 'Close', 'Datetime']
        X_clf = self.data.drop(columns=[c for c in drop_cols if c in self.data.columns])
        y_clf = self.data['trend']

        split = int(len(X_clf) * 0.8)
        X_clf_train = X_clf.iloc[:split]
        X_clf_test = X_clf.iloc[split:]

        y_clf_train = y_clf.iloc[:split]
        y_clf_test = y_clf.iloc[split:]

        print(f"RF for Classification -> Train size: {len(X_clf_train)} | Test size: {len(X_clf_test)}")
        return X_clf_train, y_clf_train, X_clf_test, y_clf_test

    def get_train_test_reg(self):
        drop_cols = ['trend', 'Close', 'Datetime']
        X_reg = self.data.drop(columns=[c for c in drop_cols if c in self.data.columns])
        y_reg = self.data['Close']

        split = int(len(X_reg) * 0.8)
        X_reg_train = X_reg.iloc[:split]
        X_reg_test = X_reg.iloc[split:]

        y_reg_train = y_reg.iloc[:split]
        y_reg_test = y_reg.iloc[split:]

        print(f"RF for Regression -> Train size: {len(X_reg_train)} | Test size: {len(X_reg_test)}")
        return X_reg_train, y_reg_train, X_reg_test, y_reg_test


In [26]:
preprocess = Preprocess(fe_data)

X_clf_train, y_clf_train, X_clf_test, y_clf_test = preprocess.get_train_test_clf()
X_reg_train, y_reg_train, X_reg_test, y_reg_test = preprocess.get_train_test_reg()

RF for Classification -> Train size: 5576 | Test size: 1395
RF for Regression -> Train size: 5576 | Test size: 1395


### Random Forest Models

The `RandomForest` class wraps both a classifier and a regressor, sharing a common hyperparameter search routine.

#### Hyperparameter Search (`find_best_params`)

`RandomizedSearchCV` is used instead of an exhaustive grid search for efficiency. It samples `n_iter=30` random combinations from the parameter grid and evaluates each using **5-fold time-series cross-validation** (`TimeSeriesSplit`), which respects the temporal ordering of data and avoids future leakage.

The parameter grid explored:

| Parameter | Values |
|---|---|
| `n_estimators` | 100, 200, 300, 500 |
| `max_depth` | 5, 10, 20, None |
| `max_features` | `'sqrt'`, `'log2'`, 0.5 |
| `min_samples_split` | 2, 5, 10 |
| `min_samples_leaf` | 1, 2, 4 |

#### Classifier (`classifier`)

Predicts the **price direction** (up = `1`, down = `0`) for the next bar. Uses `class_weight='balanced'` to handle any class imbalance in trending vs. non-trending periods. Optimized with `f1` scoring and evaluated with a full classification report.

#### Regressor (`regression`)

Predicts the **actual closing price** for the next bar. Optimized with `R2` and evaluated with MAE and R².


In [27]:
CLF_BASE = 'clf'
REG_BASE = 'reg'

CLF_SCORING = 'f1'
REG_SCORING = 'r2'

class RandomForest():
    def __init__(self):
        self.param_grid = {
            'n_estimators' : [50, 100],
            'max_depth' : [2, 5, 10, 20],
            'max_features' : ['sqrt', 'log2', 0.5],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf' : [1, 2, 4]
        }

        self.tscv = TimeSeriesSplit(n_splits=5)

        self.clf_base = RandomForestClassifier(random_state=42, class_weight='balanced')
        self.reg_base = RandomForestRegressor(random_state=42)

    def find_best_params(self, X, y, base=CLF_BASE, n_iter=10, scoring=CLF_SCORING, verbose=1):
        if base == 'clf':
            estimator = self.clf_base
        elif base == 'reg':
            estimator = self.reg_base
        else:
            raise ValueError("Base does not exist!")
        search = RandomizedSearchCV(
            estimator = estimator,
            param_distributions= self.param_grid,
            n_iter = n_iter,
            scoring = scoring,
            cv = self.tscv,
            n_jobs = -1,
            random_state = 42,
            verbose = verbose
        )

        search.fit(X, y)

        return search.best_estimator_, round(search.best_score_, 4)

    def classifier(self, X, y, X_test, y_test, verbose=0):
        best_clf, _ = self.find_best_params(X, y, base=CLF_BASE, scoring=CLF_SCORING, verbose=1)
        y_pred = best_clf.predict(X_test)

        self.clf_report = classification_report(y_test, y_pred, target_names=['Down (0)', 'Up (1)'])
        self.best_clf = best_clf

        if verbose:
            print(self.clf_report)

    def regression(self, X, y, X_test, y_test, verbose=0):
        best_reg, _ = self.find_best_params(X, y, base=REG_BASE, scoring=REG_SCORING, verbose=1)
        y_pred = best_reg.predict(X_test)

        self.reg_mae = mean_absolute_error(y_test, y_pred)
        self.reg_r2_score = r2_score(y_test, y_pred)
        self.best_reg = best_reg

        if verbose:
            print(f"MAE : {self.reg_mae:.4f}")
            print(f"R² : {self.reg_r2_score:.4f}")

    def get_best_clf(self):
        return self.best_clf

    def get_best_reg(self):
        return self.best_reg

In [28]:
RF = RandomForest()
RF.classifier(X=X_clf_train, y=y_clf_train, X_test=X_clf_test, y_test=y_clf_test, verbose=1)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
              precision    recall  f1-score   support

    Down (0)       1.00      1.00      1.00       707
      Up (1)       1.00      1.00      1.00       688

    accuracy                           1.00      1395
   macro avg       1.00      1.00      1.00      1395
weighted avg       1.00      1.00      1.00      1395



In [29]:
RF = RandomForest()
RF.regression(X=X_reg_train, y=y_reg_train, X_test=X_reg_test, y_test=y_reg_test)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


this is where the change begins

In [30]:
class TransactionType(Enum):
    BUY = "PURCHASE"
    SELL = "SALE"

class Currency(Enum):
    USD = "USD"
    EUR = "EUR"

@dataclass
class Security:
    name: str = ""
    ticker: str = ""
    currency: str = "USD"

@dataclass
class Transaction:
    type: TransactionType = None
    date: str = ""
    currency: str = "USD"
    shares: float = 0.0
    price: float = 0.0
    security: Security = None

    def __repr__(self):
        action = self.type.value if self.type else "?"
        ticker = self.security.ticker if self.security else "?"
        return (
            f"[{self.date}] {action:8s} {self.shares:.4f} x {ticker} @ {self.price:.4f}"
        )

class TradeLog:
    def __init__(self):
        self.transactions: List[Transaction] = []

    def append_trade(self, type: TransactionType, date: str, currency: str,
                     shares: float, price: float, security: Security):
        t = Transaction(
            type=type, date=date, currency=currency,
            shares=shares, price=price, security=security
        )
        self.transactions.append(t)
        return t

    def display(self):
        if not self.transactions:
            print("Trade log is empty — no trades executed.")
            return
        print(f"{'#':<4} {'Date':<22} {'Action':<10} {'Shares':<10} {'Ticker':<8} {'Price':>10}")
        print("-" * 68)
        for i, t in enumerate(self.transactions, 1):
            print(f"{i:<4} {t.date:<22} {t.type.value:<10} {t.shares:<10.4f} {t.security.ticker:<8} {t.price:>10.4f}")
        print(f"\nTotal trades: {len(self.transactions)}")


In [31]:
MIN_PREDICTED_GAIN_PCT = 0.0005  # 0.05% minimum expected gain to trigger a BUY
NUM_SHARES_TO_BUY = 1000.0

# direction
UP = 1
DOWN = 0

# position
LONG = 1
HOLD = 0

class Trade:
    def __init__(self, data):
        self.data = data
        self.preprocess = Preprocess(data)
        self.rf = RandomForest()
        self.tradelog = TradeLog()

        print("Training classifier...")
        self.train_classifier()
        print("Training regressor...")
        self.train_regression()

    def train_classifier(self):
        X_clf_train, y_clf_train, X_clf_test, y_clf_test = self.preprocess.get_train_test_clf()
        self.rf.classifier(X=X_clf_train, y=y_clf_train, X_test=X_clf_test, y_test=y_clf_test)

    def train_regression(self):
        X_reg_train, y_reg_train, X_reg_test, y_reg_test = self.preprocess.get_train_test_reg()
        self.rf.regression(X=X_reg_train, y=y_reg_train, X_test=X_reg_test, y_test=y_reg_test)

    def run_backtest(self, ticker: str = "AAPL", currency: str = "USD", verbose=1, display_tradelog=0):

        data = self.preprocess.get_data()
        drop_cols = ['trend', 'Close', 'Datetime']
        X = data.drop(columns=[c for c in drop_cols if c in data.columns])

        # Use only the latest bar for inference
        X_latest      = X.iloc[[-1]]
        current_price = float(data['Close'].iloc[-1])
        now           = str(data['Datetime'].iloc[-1])

        security = Security(name=ticker, ticker=ticker, currency=currency)

        direction = int(self.rf.best_clf.predict(X_latest)[0])
        predicted_price = float(self.rf.best_reg.predict(X_latest)[0])
        predicted_gain  = (predicted_price - current_price) / current_price

        if verbose:
          print(f"\n[{now}] Current price : {current_price:.4f}")
          print(f"[{now}] Predicted price: {predicted_price:.4f}  ({predicted_gain*100:+.3f}%)")
          print(f"[{now}] Direction      : {'UP' if direction == 1 else 'DOWN'}")
          print()

        position    = self._current_position()
        shares_held = self._shares_held()

        # BUY
        if direction == UP and predicted_gain >= MIN_PREDICTED_GAIN_PCT:
            if position == HOLD:
                shares = round(NUM_SHARES_TO_BUY / current_price, 6)
                self.tradelog.append_trade(
                    type=TransactionType.BUY,
                    currency=currency,
                    date=now,
                    shares=shares,
                    price=current_price,
                    security=security,
                )
                print(f"[{now}] Decision: BUY {shares} x {ticker} @ {current_price:.4f}")
            else:
                print(f"[{now}] Decision: HOLD — already long.")
        # SELL
        elif direction == DOWN:
            if position == 1:
                self.tradelog.append_trade(
                    type=TransactionType.SELL,
                    currency=currency,
                    date=now,
                    shares=shares_held,
                    price=current_price,
                    security=security,
                )
                print(f"[{now}] Decision: SELL {shares_held} x {ticker} @ {current_price:.4f}")
            else:
                print(f"[{now}] Decision: HOLD — already flat.")
        else:
            print(f"[{now}] Decision: HOLD — conditions not met.")

        if display_tradelog:
          print("\n=== TRADE LOG ===")
          self.tradelog.display()

    def _current_position(self) -> int:
        return LONG if self._shares_held() > 0 else HOLD

    def _shares_held(self) -> float:
        net = 0.0
        for t in self.tradelog.transactions:
            if t.type == TransactionType.BUY:
                net += t.shares
            elif t.type == TransactionType.SELL:
                net -= t.shares
        return max(net, 0.0)

In [32]:
trade = Trade(fe_data)
trade.run_backtest(ticker="AAPL", display_tradelog=1)

Training classifier...
RF for Classification -> Train size: 5576 | Test size: 1395
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Training regressor...
RF for Regression -> Train size: 5576 | Test size: 1395
Fitting 5 folds for each of 10 candidates, totalling 50 fits

[2026-05-15 19:59:00+00:00] Current price : 300.2200
[2026-05-15 19:59:00+00:00] Predicted price: 294.5350  (-1.894%)
[2026-05-15 19:59:00+00:00] Direction      : DOWN

[2026-05-15 19:59:00+00:00] Decision: HOLD — already flat.

=== TRADE LOG ===
Trade log is empty — no trades executed.


In [33]:
# testing
fake_security = Security(name="Apple Inc.", ticker="AAPL", currency="USD")

trade.tradelog.append_trade(
    type=TransactionType.BUY,
    date="2024-01-10 09:31:00",
    currency="USD",
    shares=5.2631,
    price=190.00,
    security=fake_security,
)

print("=== Pre-seeded trade log ===")
trade.tradelog.display()


=== Pre-seeded trade log ===
#    Date                   Action     Shares     Ticker        Price
--------------------------------------------------------------------
1    2024-01-10 09:31:00    PURCHASE   5.2631     AAPL       190.0000

Total trades: 1


In [34]:
print("\n--- Now running backtest (new trades will be appended) ---\n")
trade.run_backtest(ticker="AAPL")


--- Now running backtest (new trades will be appended) ---


[2026-05-15 19:59:00+00:00] Current price : 300.2200
[2026-05-15 19:59:00+00:00] Predicted price: 294.5350  (-1.894%)
[2026-05-15 19:59:00+00:00] Direction      : DOWN

[2026-05-15 19:59:00+00:00] Decision: SELL 5.2631 x AAPL @ 300.2200
